# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', getattr(metadata, '@id', 'N/A'))}\n\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id` fields.

We'll display all record sets in this dataset, along with their field IDs and names.

In [ ]:
# List all record sets and their fields, showing @id and their field @id's
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
if not record_sets:
    print("No record sets were found in the metadata.")
else:
    for rec in record_sets:
        rec_id = getattr(rec, "@id", None)
        rec_name = getattr(rec, "name", "")
        print(f"RecordSet @id: {rec_id} | name: {rec_name}")
        if hasattr(rec, 'field'):
            fields = rec.field
            for field in fields:
                field_id = getattr(field, "@id", None)
                field_name = getattr(field, "name", "")
                print(f"    Field @id: {field_id} | name: {field_name}")

For demonstration, let's print some examples for each record in the first available RecordSet, referencing by `@id`. If the dataset provides multiple record sets, select one for subsequent analysis.

In [ ]:
# Show sample records from the first record set
if not record_sets:
    print("No record sets available in the dataset.")
else:
    first_rec = record_sets[0]
    first_rec_id = getattr(first_rec, "@id")
    print(f"Sample records from RecordSet @id: {first_rec_id}")
    for idx, row in enumerate(dataset.records(record_set=first_rec_id)):
        print(row)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing each by their `@id`. This enables inspection and EDA across all record sets.

In [ ]:
dataframes = {}

if not record_sets:
    print("No record sets available for extraction.")
else:
    # Prepare list of record set IDs
    record_sets_ids = [getattr(rec, "@id") for rec in record_sets]

    for rec_id in record_sets_ids:
        print(f"Loading records for RecordSet @id: {rec_id}")
        try:
            rows = list(dataset.records(record_set=rec_id))
            if rows:
                df = pd.DataFrame(rows)
                print(f"  Shape: {df.shape} | Columns: {df.columns.tolist()}")
            else:
                df = pd.DataFrame()
                print(f"  No records found in this RecordSet.")
            dataframes[rec_id] = df
        except Exception as e:
            print(f"  Error loading records: {e}")

    # Example: Show columns and first few rows of the first record set with data
    first_with_data = None
    for rec_id, df in dataframes.items():
        if not df.empty:
            first_with_data = rec_id
            break

    if first_with_data:
        print(f"\nColumns in RecordSet {first_with_data}: \n{dataframes[first_with_data].columns.tolist()}")
        display(dataframes[first_with_data].head())
    else:
        print("No record set contains data to show.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by categorical attributes. We'll use the `@id` fields obtained above to reference the correct fields.

In [ ]:
# For demonstration, select the first DataFrame with numeric fields
import numpy as np

if not dataframes:
    print("No DataFrames available for EDA.")
elif not first_with_data:
    print("No RecordSet with data found for EDA.")
else:
    df = dataframes[first_with_data]
    # Find a numeric field in the DataFrame - prefer common scientific identifiers
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        print("No numeric field found in this record set.")
    else:
        print(f"Using numeric field (@id): {numeric_field}")
        threshold = df[numeric_field].mean()  # Example threshold: mean of the column
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        normalized_field = f"{numeric_field}_normalized"
        filtered_df[normalized_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, normalized_field]].head())

        # Try grouping by a likely categorical field
        group_field = None
        for col in df.columns:
            if (col != numeric_field and df[col].dtype == 'object' and df[col].nunique() < 20):
                group_field = col
                break
        if group_field:
            print(f"\nGrouped data by {group_field} (mean of {numeric_field} and normalized):")
            grouped_df = filtered_df.groupby(group_field)[[numeric_field, normalized_field]].mean()
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping was found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the effect of normalization using matplotlib and seaborn. Relationships between key fields can also be visualized where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not first_with_data or not numeric_field:
    print("No numeric data available for plotting.")
else:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)

    plt.subplot(1,2,2)
    if normalized_field in filtered_df.columns:
        sns.histplot(filtered_df[normalized_field], bins=20, kde=True)
        plt.title(f"Normalized {numeric_field} in Filtered Records")
        plt.xlabel(normalized_field)
    else:
        plt.text(0.5,0.5, 'No normalized field to plot', ha='center')

    plt.tight_layout()
    plt.show()

    # Optional: Visualize relationship between numeric_field and group_field if available
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
- Loaded dataset metadata and record sets using `mlcroissant`, referencing all entities by their `@id` fields for reproducibility and schema consistency.
- Explored the list of record sets and fields. Tabular data for each record set was loaded for analysis.
- Performed basic exploratory data analysis (EDA): filtered a numeric field, normalized the values, and grouped by a suitable categorical field if found.
- Visualized the raw and normalized numeric distributions. 

This process can be adapted to more complex analysis or more specific field selection, and extended for further downstream ML tasks, while operating entirely through the Croissant schema specification.